# LogiScan Phase 4: Final Production Retraining (10.7K)

This is the official production retraining notebook for the 24-class Phase 4 model.

### Production Specs:
1. **Dataset**: V3.1 (10,786 samples) - Bridged recall gaps and boosted boundary classes.
2. **Stabilization**: GradScaler + Clipping + Sqrt-Smoothing applied.
3. **Architecture**: DeBERTa-v3-small (6-layer, high efficiency).
4. **Goal**: Target Macro-F1 > 0.85.

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn tqdm sentencepiece

In [ ]:
import json

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"READY: Training on {DEVICE}")

### 1. Data Ingestion

In [ ]:
with open("unified_training_data.json") as f:
    data = json.load(f)

texts = [d["text"] for d in data]
label_list = sorted(list(set(d["fallacy"] for d in data)))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
labels = [label2id[d["fallacy"]] for d in data]

# Production Weighting: Sqrt Smoothing
label_counts = np.bincount(labels)
weights = 1.0 / (np.sqrt(label_counts) + 1e-6)
weights = weights / weights.sum() * len(label_list)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)
print(f"Samples: {len(texts)} | Classes: {len(label_list)}")

In [ ]:
class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "label": torch.tensor(self.labels[idx], dtype=torch.long)}

### 2. Model Setup

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id).to(DEVICE)

train_loader = DataLoader(FallacyDataset(X_train, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(FallacyDataset(X_val, y_val, tokenizer), batch_size=32)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda') if DEVICE.type == "cuda" else None
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=len(train_loader)//2, num_training_steps=len(train_loader)*6)

### 3. Production Training Loop

In [ ]:
best_f1, loss_fn = 0, nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(6):
    model.train()
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress:
        optimizer.zero_grad()
        ids, mask, lbls = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE), batch["label"].to(DEVICE)

        if scaler:
            with torch.amp.autocast('cuda'):
                loss = loss_fn(model(ids, attention_mask=mask).logits, lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = loss_fn(model(ids, attention_mask=mask).logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        scheduler.step()
        progress.set_postfix({"loss": f"{loss.item():.4f}"})

    # Validation
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for b in val_loader:
            out = model(b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
            all_p.extend(torch.argmax(out.logits, 1).cpu().numpy())
            all_l.extend(b["label"].numpy())

    f1 = f1_score(all_l, all_p, average="macro")
    print(f"\nEpoch {epoch+1} Macro-F1: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("phase4_final_model")
        tokenizer.save_pretrained("phase4_final_model")
        print("✅ NEW BEST MODEL SAVED")
    print(classification_report(all_l, all_p, target_names=label_list))

### 4. Package for Deployment

In [ ]:
!zip -r phase4_final_model.zip phase4_final_model
from google.colab import files

files.download("phase4_final_model.zip")